# Import Modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Process the data

## 22. Statlog Heart Disease

In [ ]:
df_statlog = pd.read_csv('data/22-statlog.csv')
df_statlog.head()

## 23. Coronary Heart Disease

In [ ]:
df_coronary = pd.read_csv('data/23-chd.csv')
df_coronary.head()

---
# EDA — Exploratory Data Analysis

## Shape & Basic Info

In [ ]:
print('=== 22. Statlog Heart Disease ===')
print(f'  Rows: {df_statlog.shape[0]}   Columns: {df_statlog.shape[1]}')
print()
df_statlog.info()

In [ ]:
print('=== 23. Coronary Heart Disease ===')
print(f'  Rows: {df_coronary.shape[0]}   Columns: {df_coronary.shape[1]}')
print()
df_coronary.info()

## Descriptive Statistics

In [ ]:
print('=== 22. Statlog — descriptive stats ===')
df_statlog.describe().round(2)

In [ ]:
print('=== 23. Coronary — descriptive stats ===')
df_coronary.describe().round(2)

## Missing Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, df, title in zip(
    axes,
    [df_statlog, df_coronary],
    ['22. Statlog Heart Disease', '23. Coronary Heart Disease']
):
    null_pct = df.isnull().mean() * 100
    null_pct = null_pct[null_pct > 0]
    if null_pct.empty:
        ax.text(0.5, 0.5, 'No missing values', ha='center', va='center',
                fontsize=13, transform=ax.transAxes)
    else:
        null_pct.sort_values().plot(kind='barh', ax=ax, color='#d98080')
        ax.set_xlabel('% missing')
        for bar, val in zip(ax.patches, null_pct.sort_values()):
            ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}%', va='center', fontsize=9)
    ax.set_title(title)

plt.tight_layout()
plt.show()

print('Statlog nulls:')
print(df_statlog.isnull().sum())
print()
print('Coronary nulls:')
print(df_coronary.isnull().sum())

## Class Balance (Target Variable)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Statlog: heart-disease  1=absent, 2=present
vc_s = df_statlog['heart-disease'].map({1: 'Absent (1)', 2: 'Present (2)'}).value_counts()
axes[0].pie(vc_s, labels=vc_s.index, autopct='%1.1f%%',
            colors=['#7bafd4', '#d98080'], startangle=90)
axes[0].set_title('22. Statlog — heart-disease')

# Coronary: chd  0=absent, 1=present
vc_c = df_coronary['chd'].map({0: 'Absent (0)', 1: 'Present (1)'}).value_counts()
axes[1].pie(vc_c, labels=vc_c.index, autopct='%1.1f%%',
            colors=['#7bafd4', '#d98080'], startangle=90)
axes[1].set_title('23. Coronary — chd')

plt.tight_layout()
plt.show()

print('Statlog class counts:\n', df_statlog['heart-disease'].value_counts().rename({1:'Absent',2:'Present'}))
print()
print('Coronary class counts:\n', df_coronary['chd'].value_counts().rename({0:'Absent',1:'Present'}))

## Feature Distributions — 22. Statlog

In [ ]:
# Continuous features
cont_s = ['age', 'rest-bp', 'serum-chol', 'max-heart-rate', 'oldpeak']

fig, axes = plt.subplots(1, len(cont_s), figsize=(18, 3.5))
for ax, col in zip(axes, cont_s):
    for label, grp in df_statlog.groupby('heart-disease')[col]:
        grp.dropna().plot(kind='kde', ax=ax,
                          label=('Absent' if label == 1 else 'Present'),
                          color=('#7bafd4' if label == 1 else '#d98080'))
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

fig.suptitle('22. Statlog — continuous features by disease status', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical / binary features
cat_s = ['sex', 'chest-pain', 'fasting-blood-sugar',
         'electrocardiographic', 'angina', 'slope', 'major-vessels', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()

for ax, col in zip(axes, cat_s):
    ct = pd.crosstab(
        df_statlog[col],
        df_statlog['heart-disease'].map({1: 'Absent', 2: 'Present'})
    )
    ct.plot(kind='bar', ax=ax, color=['#7bafd4', '#d98080'],
            edgecolor='none', legend=True, rot=0)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=7)

fig.suptitle('22. Statlog — categorical features by disease status', y=1.01)
plt.tight_layout()
plt.show()

## Feature Distributions — 23. Coronary

In [ ]:
cont_c = ['sbp', 'tobacco', 'ldl', 'adiposity', 'typea', 'obesity', 'alcohol', 'age']

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()

for ax, col in zip(axes, cont_c):
    for label, grp in df_coronary.groupby('chd')[col]:
        grp.plot(kind='kde', ax=ax,
                 label=('Absent' if label == 0 else 'Present'),
                 color=('#7bafd4' if label == 0 else '#d98080'))
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

fig.suptitle('23. Coronary — continuous features by CHD status', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# famhist is the only categorical feature
ct = pd.crosstab(
    df_coronary['famhist'],
    df_coronary['chd'].map({0: 'Absent', 1: 'Present'})
)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ct.plot(kind='bar', ax=axes[0], color=['#7bafd4', '#d98080'], rot=0, edgecolor='none')
axes[0].set_title('famhist — counts')
axes[0].set_xlabel('')

ct_pct.plot(kind='bar', ax=axes[1], color=['#7bafd4', '#d98080'], rot=0, edgecolor='none')
axes[1].set_title('famhist — % within group')
axes[1].set_ylabel('%')
axes[1].set_xlabel('')

plt.suptitle('23. Coronary — family history vs CHD', y=1.02)
plt.tight_layout()
plt.show()

## Correlation Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Statlog — encode target temporarily for correlation
tmp_s = df_statlog.copy()
tmp_s['heart-disease'] = (tmp_s['heart-disease'] == 2).astype(int)
corr_s = tmp_s.corr()

mask_s = np.triu(np.ones_like(corr_s, dtype=bool))
sns.heatmap(corr_s, mask=mask_s, ax=axes[0],
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.4)
axes[0].set_title('22. Statlog — correlation matrix', fontsize=11)

# Coronary — encode famhist
tmp_c = df_coronary.copy()
tmp_c['famhist'] = (tmp_c['famhist'] == 'Present').astype(int)
corr_c = tmp_c.corr()

mask_c = np.triu(np.ones_like(corr_c, dtype=bool))
sns.heatmap(corr_c, mask=mask_c, ax=axes[1],
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            linewidths=0.4)
axes[1].set_title('23. Coronary — correlation matrix', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target — sorted bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

corr_target_s = corr_s['heart-disease'].drop('heart-disease').sort_values()
colors_s = ['#d98080' if v > 0 else '#7bafd4' for v in corr_target_s]
corr_target_s.plot(kind='barh', ax=axes[0], color=colors_s, edgecolor='none')
axes[0].axvline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].set_title('22. Statlog — correlation with heart-disease')
axes[0].set_xlabel('Pearson r')

corr_target_c = corr_c['chd'].drop('chd').sort_values()
colors_c = ['#d98080' if v > 0 else '#7bafd4' for v in corr_target_c]
corr_target_c.plot(kind='barh', ax=axes[1], color=colors_c, edgecolor='none')
axes[1].axvline(0, color='gray', linewidth=0.8, linestyle='--')
axes[1].set_title('23. Coronary — correlation with chd')
axes[1].set_xlabel('Pearson r')

plt.tight_layout()
plt.show()

---
# Data Processing

Steps applied to both datasets:
1. **Handle missing values** — median imputation for numerical columns
2. **Encode categorical features** — binary mapping
3. **Standardise the target** — both become `0 = no disease, 1 = disease`
4. **Normalise continuous features** — MinMax scaling to [0, 1]

## 22. Statlog — Processing

In [ ]:
# ── Step 1: Check & impute missing values ─────────────────────────────
print('Missing values BEFORE imputation:')
print(df_statlog.isnull().sum()[df_statlog.isnull().sum() > 0])

df_s = df_statlog.copy()

# major-vessels and thal have rare NaN — fill with median
for col in ['major-vessels', 'thal']:
    median_val = df_s[col].median()
    df_s[col] = df_s[col].fillna(median_val)
    print(f'  Imputed "{col}" with median = {median_val}')

print('\nMissing values AFTER imputation:', df_s.isnull().sum().sum())

In [ ]:
# ── Step 2: Encode target — 1=absent → 0, 2=present → 1 ──────────────
df_s['heart_disease'] = (df_s['heart-disease'] == 2).astype(int)
df_s = df_s.drop(columns=['heart-disease'])

print('Target distribution after encoding:')
print(df_s['heart_disease'].value_counts().rename({0:'No disease (0)', 1:'Disease (1)'}))

In [ ]:
# ── Step 3: Re-encode thal (3/6/7 are nominal codes, not ordinal) ─────
#   3 = normal       → 0
#   6 = fixed defect → 1
#   7 = reversible   → 2
df_s['thal'] = df_s['thal'].map({3.0: 0, 6.0: 1, 7.0: 2})

print('thal after mapping:')
print(df_s['thal'].value_counts().sort_index())

In [ ]:
# ── Step 4: Normalise continuous features ─────────────────────────────
cont_s = ['age', 'rest-bp', 'serum-chol', 'max-heart-rate', 'oldpeak']

scaler_s = MinMaxScaler()
df_s[cont_s] = scaler_s.fit_transform(df_s[cont_s])

print('Continuous columns after MinMax scaling (should be [0, 1]):')
df_s[cont_s].agg(['min', 'max']).round(4)

In [ ]:
print('=== 22. Statlog — final processed shape:', df_s.shape, '===')
print('Columns:', df_s.columns.tolist())
df_s.head()

## 23. Coronary — Processing

In [ ]:
# ── Step 1: Check missing values ──────────────────────────────────────
print('Missing values:')
null_counts = df_coronary.isnull().sum()
print(null_counts)

df_c = df_coronary.copy()
if null_counts.sum() == 0:
    print('\n→ No missing values — no imputation needed.')

In [ ]:
# ── Step 2: Encode famhist (Present/Absent → 1/0) ─────────────────────
print('famhist BEFORE:', df_c['famhist'].unique())

df_c['famhist'] = (df_c['famhist'] == 'Present').astype(int)

print('famhist AFTER: ', df_c['famhist'].unique())
print('Distribution:', df_c['famhist'].value_counts().to_dict())

In [ ]:
# ── Step 3: Verify target (chd is already 0/1) ────────────────────────
print('Target distribution (chd):')
print(df_c['chd'].value_counts().rename({0:'No CHD (0)', 1:'CHD (1)'}))

In [ ]:
# ── Step 4: Normalise continuous features ─────────────────────────────
# tobacco and alcohol are right-skewed — log-transform before scaling
for col in ['tobacco', 'alcohol']:
    df_c[col] = np.log1p(df_c[col])  # log(1 + x) handles zeros
    print(f'  Applied log1p to "{col}" (was right-skewed)')

cont_c = ['sbp', 'tobacco', 'ldl', 'adiposity', 'typea', 'obesity', 'alcohol', 'age']
scaler_c = MinMaxScaler()
df_c[cont_c] = scaler_c.fit_transform(df_c[cont_c])

print('\nContinuous columns after scaling (should be [0, 1]):')
df_c[cont_c].agg(['min', 'max']).round(4)

In [ ]:
print('=== 23. Coronary — final processed shape:', df_c.shape, '===')
print('Columns:', df_c.columns.tolist())
df_c.head()

## Before vs After — Distribution Check

In [ ]:
# Visual check: raw vs processed for key continuous cols
check_cols_s = ['age', 'rest-bp', 'serum-chol']
check_cols_c = ['sbp', 'ldl', 'age']

fig, axes = plt.subplots(2, 6, figsize=(22, 6))

for i, col in enumerate(check_cols_s):
    df_statlog[col].dropna().plot(kind='hist', bins=25, ax=axes[0, i],
                                  color='#7bafd4', edgecolor='none', density=True)
    axes[0, i].set_title(f'Statlog raw — {col}', fontsize=9)

    df_s[col].plot(kind='hist', bins=25, ax=axes[0, i+3],
                   color='#6aab8a', edgecolor='none', density=True)
    axes[0, i+3].set_title(f'Statlog scaled — {col}', fontsize=9)

for i, col in enumerate(check_cols_c):
    df_coronary[col].plot(kind='hist', bins=25, ax=axes[1, i],
                          color='#7bafd4', edgecolor='none', density=True)
    axes[1, i].set_title(f'Coronary raw — {col}', fontsize=9)

    df_c[col].plot(kind='hist', bins=25, ax=axes[1, i+3],
                   color='#6aab8a', edgecolor='none', density=True)
    axes[1, i+3].set_title(f'Coronary scaled — {col}', fontsize=9)

fig.suptitle('Distribution before (blue) vs after scaling (green)', y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

## Save Processed Datasets

In [ ]:
df_s.to_csv('data/22-statlog-processed.csv', index=False)
df_c.to_csv('data/23-coronary-processed.csv', index=False)

print('Saved:')
print('  data/22-statlog-processed.csv  →', df_s.shape)
print('  data/23-coronary-processed.csv →', df_c.shape)

---
# Join — Merge Both Datasets

Both datasets share  and  and a binary disease target.
All other columns are dataset-specific — they will be **kept as ** for the dataset that does not have them.

| Column | Statlog | Coronary |
|---|---|---|
|  | ✓ | ✓ |
|  |  |  |
|  |  |  |
| , , , , , , , , , ,  | ✓ | — |
| , , , , , ,  | — | ✓ |

## Align Statlog to Unified Schema

In [ ]:
merged_statlog = pd.DataFrame()

# Set data columns first so the index is established from df_s
merged_statlog["age"]            = df_s["age"]
merged_statlog["systolic_bp"]    = df_s["rest-bp"]
merged_statlog["sex"]            = df_s["sex"]
merged_statlog["chest_pain"]     = df_s["chest-pain"]
merged_statlog["serum_chol"]     = df_s["serum-chol"]
merged_statlog["fbs"]            = df_s["fasting-blood-sugar"]
merged_statlog["ecg"]            = df_s["electrocardiographic"]
merged_statlog["max_heart_rate"] = df_s["max-heart-rate"]
merged_statlog["angina"]         = df_s["angina"]
merged_statlog["oldpeak"]        = df_s["oldpeak"]
merged_statlog["slope"]          = df_s["slope"]
merged_statlog["major_vessels"]  = df_s["major-vessels"]
merged_statlog["thal"]           = df_s["thal"]
for col in ["tobacco","ldl","adiposity","family_history","typea","obesity","alcohol"]:
    merged_statlog[col] = np.nan
merged_statlog["has_disease"]    = df_s["heart_disease"]
# source set last to broadcast over established index
merged_statlog["source"]         = "statlog"

print("Statlog aligned:", merged_statlog.shape)
merged_statlog.head(3)

## Align Coronary to Unified Schema

In [ ]:
merged_coronary = pd.DataFrame()

merged_coronary["age"]            = df_c["age"]
merged_coronary["systolic_bp"]    = df_c["sbp"]
for col in ["sex","chest_pain","serum_chol","fbs","ecg","max_heart_rate",
             "angina","oldpeak","slope","major_vessels","thal"]:
    merged_coronary[col] = np.nan
merged_coronary["tobacco"]        = df_c["tobacco"]
merged_coronary["ldl"]            = df_c["ldl"]
merged_coronary["adiposity"]      = df_c["adiposity"]
merged_coronary["family_history"] = df_c["famhist"]
merged_coronary["typea"]          = df_c["typea"]
merged_coronary["obesity"]        = df_c["obesity"]
merged_coronary["alcohol"]        = df_c["alcohol"]
merged_coronary["has_disease"]    = df_c["chd"]
merged_coronary["source"]         = "coronary"

print("Coronary aligned:", merged_coronary.shape)
merged_coronary.head(3)

## Concatenate

In [ ]:
df_merged = pd.concat([merged_statlog, merged_coronary], ignore_index=True)

n_s = (df_merged["source"]=="statlog").sum()
n_c = (df_merged["source"]=="coronary").sum()
print("Merged shape:", df_merged.shape)
print("  Statlog rows :", n_s)
print("  Coronary rows:", n_c)
print("Target balance:")
print(df_merged["has_disease"].value_counts().rename({0:"No disease",1:"Disease"}))
df_merged.head()

## Null Coverage Map

In [ ]:
null_pct = df_merged.drop(columns=["source","has_disease"]).isnull().mean() * 100

fig, ax = plt.subplots(figsize=(11, 5))
sorted_vals = null_pct.sort_values(ascending=True)
bar_colors = ["#d98080" if v > 50 else "#7bafd4" if v > 0 else "#6aab8a" for v in sorted_vals]
sorted_vals.plot(kind="barh", ax=ax, color=bar_colors, edgecolor="none")
ax.axvline(63.1, color="#d98080", linestyle="--", linewidth=0.8, alpha=0.7)
ax.axvline(36.9, color="#7bafd4", linestyle="--", linewidth=0.8, alpha=0.7)
ax.set_xlabel("% null in merged dataset")
ax.set_title("Null coverage per column — expected by design")
import matplotlib.patches as mpatches
p1 = mpatches.Patch(color="#6aab8a", label="Shared (both datasets)")
p2 = mpatches.Patch(color="#7bafd4", label="Statlog-only (null for coronary rows)")
p3 = mpatches.Patch(color="#d98080", label="Coronary-only (null for statlog rows)")
ax.legend(handles=[p1,p2,p3], fontsize=8)
plt.tight_layout()
plt.show()

print("Null % per column:")
print(null_pct.round(1).to_string())

## Disease Balance by Source

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, src, title in zip(
    axes[:2],
    ["statlog", "coronary"],
    ["22. Statlog", "23. Coronary"]
):
    vc = df_merged[df_merged["source"]==src]["has_disease"].value_counts().rename({0:"No",1:"Yes"})
    ax.pie(vc, labels=vc.index, autopct="%1.1f%%",
           colors=["#7bafd4","#d98080"], startangle=90)
    ax.set_title(title + " has_disease")

vc_all = df_merged["has_disease"].value_counts().rename({0:"No",1:"Yes"})
axes[2].pie(vc_all, labels=vc_all.index, autopct="%1.1f%%",
            colors=["#7bafd4","#d98080"], startangle=90)
axes[2].set_title("Combined has_disease")

plt.suptitle("Disease balance by dataset and overall", y=1.02)
plt.tight_layout()
plt.show()

## Shared Features — Cross-Dataset Distribution Check

 and  come from both datasets. Check they are on the same scale after normalization.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ["age", "systolic_bp"]):
    for src, color in [("statlog","#7bafd4"),("coronary","#d98080")]:
        df_merged[df_merged["source"]==src][col].plot(
            kind="kde", ax=ax, label=src, color=color, linewidth=2)
    ax.set_title(col + " — distribution by source")
    ax.set_xlabel("normalised value [0-1]")
    ax.legend()

plt.suptitle("Shared columns — sanity check after normalization", y=1.02)
plt.tight_layout()
plt.show()

print("age stats by source:")
print(df_merged.groupby("source")["age"].describe().round(3))
print("systolic_bp stats by source:")
print(df_merged.groupby("source")["systolic_bp"].describe().round(3))

## Save Merged Dataset

In [ ]:
df_merged.to_csv("data/merged.csv", index=False)

n_statlog = (df_merged["source"]=="statlog").sum()
n_coronary = (df_merged["source"]=="coronary").sum()
n_disease = df_merged["has_disease"].sum()
n_healthy = (df_merged["has_disease"]==0).sum()
pct_disease = df_merged["has_disease"].mean() * 100
pct_healthy = 100 - pct_disease

print("Saved: data/merged.csv")
print(f"  Shape     : {df_merged.shape}")
print(f"  Rows      : {len(df_merged)} ({n_statlog} statlog + {n_coronary} coronary)")
print(f"  Columns   : {df_merged.shape[1]}")
print(f"  Disease=1 : {n_disease} ({pct_disease:.1f}%)")
print(f"  Disease=0 : {n_healthy} ({pct_healthy:.1f}%)")
print()
print(df_merged.dtypes)
